In [5]:
# SLR MBGD From Scratch
import numpy as np
import pandas as pd


class SLR_MBGD:

    # =====================================================
    # CONSTRUCTOR
    # =====================================================
    def __init__(self, learning_rate=0.001, epochs=1000, batch_size=16):

        self.learning_rate = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size

        # Parameters
        self.coef_ = 0
        self.intercept_ = 0

    # =====================================================
    # TRAIN TEST SPLIT
    # =====================================================
    def train_test_split(self, X, y, test_size=0.2, random_state=None):

        X = np.array(X)
        y = np.array(y)

        n = len(X)

        indices = np.arange(n)

        np.random.seed(random_state)
        np.random.shuffle(indices)

        X = X[indices]
        y = y[indices]

        split_index = int(n * (1 - test_size))

        X_train = X[:split_index]
        X_test = X[split_index:]

        y_train = y[:split_index]
        y_test = y[split_index:]

        return X_train, X_test, y_train, y_test

    # =====================================================
    # FIT MODEL USING MINI BATCH GRADIENT DESCENT
    # =====================================================
    def fit(self, X, y):

        X = np.array(X).reshape(-1)
        y = np.array(y)

        n = len(X)

        for epoch in range(self.epochs):

            # Shuffle data every epoch
            indices = np.arange(n)
            np.random.shuffle(indices)

            X = X[indices]
            y = y[indices]

            # Mini batches
            for i in range(0, n, self.batch_size):

                X_batch = X[i:i + self.batch_size]
                y_batch = y[i:i + self.batch_size]

                batch_n = len(X_batch)

                # Predictions
                y_pred = self.intercept_ + self.coef_ * X_batch

                # Gradients
                d_intercept = (-2 / batch_n) * np.sum(y_batch - y_pred)
                d_coef = (-2 / batch_n) * np.sum(X_batch * (y_batch - y_pred))

                # Update Parameters
                self.intercept_ = self.intercept_ - self.learning_rate * d_intercept
                self.coef_ = self.coef_ - self.learning_rate * d_coef

    # =====================================================
    # PREDICTION
    # =====================================================
    def predict(self, X):

        X = np.array(X).reshape(-1)

        y_pred = self.intercept_ + self.coef_ * X

        return y_pred

    # =====================================================
    # EVALUATION METRICS
    # =====================================================
    def evaluation(self, y_true, y_pred, X_test):

        y_true = np.array(y_true)
        y_pred = np.array(y_pred)

        # MSE
        mse = np.mean((y_true - y_pred) ** 2)

        # RMSE
        rmse = np.sqrt(mse)

        # MAE
        mae = np.mean(np.abs(y_true - y_pred))

        # R2 Score
        ss_total = np.sum((y_true - np.mean(y_true)) ** 2)
        ss_residual = np.sum((y_true - y_pred) ** 2)

        r2 = 1 - (ss_residual / ss_total)

        # Adjusted R2
        n = len(y_true)
        p = X_test.shape[1] if len(X_test.shape) > 1 else 1

        adjusted_r2 = 1 - ((1 - r2) * (n - 1) / (n - p - 1))

        # Print Metrics
        print("MSE         :", mse)
        print("RMSE        :", rmse)
        print("MAE         :", mae)
        print("R2 Score    :", r2)
        print("Adjusted R2 :", adjusted_r2)


# =====================================================
# EXAMPLE USAGE
# =====================================================

# Dataset
data = pd.read_csv("placement.csv")

# Features and Target
X = data[['CGPA']]
y = data['Package_LPA']

# Model
model = SLR_MBGD(
    learning_rate=0.001,
    epochs=1000,
    batch_size=16
)

# Split
X_train, X_test, y_train, y_test = model.train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Train
model.fit(X_train, y_train)

# Prediction
y_pred = model.predict(X_test)

# Parameters
print("Intercept :", model.intercept_)
print("Slope     :", model.coef_)

# Evaluation
model.evaluation(y_test, y_pred, X_test)


# =====================================================
# WHY MBGD DIFFERS FROM SGD
# =====================================================

# SGD:
# Updates parameters after every single sample.

# MBGD:
# Updates parameters after a mini batch of samples.

# Advantages of MBGD:
# 1. More stable than SGD
# 2. Faster than Batch GD
# 3. Better GPU utilization
# 4. Most commonly used in Deep Learning

# Disadvantages:
# 1. Requires batch size tuning
# 2. Still approximate optimization


Intercept : -0.4110783604909751
Slope     : 0.813571870192979
MSE         : 0.05323943616660512
RMSE        : 0.2307367247895426
MAE         : 0.19398306178654676
R2 Score    : 0.9550369276995989
Adjusted R2 : 0.9527887740845788
